In [13]:
# Instalasi dan impor pustaka yang dibutuhkan
# (Jalankan cell ini jika dependensi belum terpasang)
# %pip install --quiet numpy sympy plotly ipywidgets

# Penyelesaian, Proyeksi Sistem, dan Visualisasi 3D Interaktif Medan & Potensial Bola Berongga

**Deskripsi Soal Fisika:**
> Sebuah bola pejal berjejari $R_1$ memiliki rapat muatan volumetrik seragam $\rho$ di seluruh volumenya, kecuali pada suatu rongga berbentuk bola kecil berjejari $R_2$ yang terletak pada jarak $a$ dari pusat bola besar ($a + R_2 < R_1$).
>
> 1. Tentukan medan listrik $\vec{E}$ di pusat bola berongga ($r = 0$).
> 2. Tentukan potensial listrik $V$ di pusat bola berongga ($r = 0$).
>
> *Catatan Tambahan:* Sesuai formulasi standar elektrodinamika, notebook ini juga menghitung nilai medan dan potensial di **pusat rongga** (titik $\vec{r} = a\hat{x}$), membuktikan sifat homogenitas medan di dalam rongga, serta memproyeksikan sistem geometri dan hasil medan/potensial secara **3D interaktif**.

---
### Prinsip Superposisi (Superposition Principle)
Sistem bola dengan rongga kosong ekuivalen secara elektrostatik dengan superposisi dua bola padat:
1. **Bola 1 (Bola Utama)**: Bola pejal utuh berjari-jari $R_1$, berpusat di origin $\vec{0}$, dengan rapat muatan $+\rho$.
2. **Bola 2 (Bola Rongga Negatif)**: Bola pejal berjari-jari $R_2$, berpusat di $\vec{a} = a\hat{x}$, dengan rapat muatan $-\rho$.

Total medan dan potensial di sembarang titik $\vec{r}$ adalah:
$$\vec{E}(\vec{r}) = \vec{E}_1(\vec{r}) + \vec{E}_2(\vec{r})$$
$$V(\vec{r}) = V_1(\vec{r}) + V_2(\vec{r})$$

## 1. Proyeksi Sistem yang Dimaksud pada Soal (Diagram Skematik 2D Interaktif Plotly)

Sebelum melakukan perhitungan matematis, mari kita proyeksikan geometri penampang melintang sistem pada bidang $xy$ ($z = 0$) menggunakan visualisasi interaktif **Plotly 2D**. Diagram ini memperlihatkan dimensi $R_1$, $R_2$, jarak offset pergeseran $\vec{a}$, pusat koordinat bola $O(0,0)$, pusat rongga $C(a,0)$, dan arah vektor medan listrik hasil perhitungan ($\vec{E}_{pusat}$ dan $\vec{E}_{rongga}$).

In [14]:
import numpy as np
import plotly.graph_objects as go

# Parameter geometri default untuk proyeksi
R1_def = 3.0   # Jari-jari bola utama pejal
R2_def = 1.0   # Jari-jari rongga bola
a_def  = 1.4   # Jarak pusat bola utama ke pusat rongga
rho_def = 1.0  # Rapat muatan seragam
eps0_def = 1.0 # Permitivitas ruang hampa (relatif)

def plot_proyeksi_sistem_2d(R1=R1_def, R2=R2_def, a=a_def, rho=rho_def, eps0=eps0_def):
    """Membuat proyeksi penampang 2D sistem bola berongga dengan anotasi fisik lengkap menggunakan Plotly."""
    theta = np.linspace(0, 2*np.pi, 300)
    
    # Lingkaran bola pejal luar dan rongga dalam
    x_out, y_out = R1 * np.cos(theta), R1 * np.sin(theta)
    x_cav, y_cav = a + R2 * np.cos(theta), R2 * np.sin(theta)
    
    fig = go.Figure()
    
    # 1. Arsiran bahan bola pejal bermuatan positif (+rho)
    fig.add_trace(go.Scatter(
        x=x_out, y=y_out,
        mode='lines',
        fill='toself',
        fillcolor='rgba(191, 219, 254, 0.65)',
        line=dict(color='#1d4ed8', width=2.5),
        name=f'Bahan Bola Pejal (+ρ), R<sub>1</sub>={R1}',
        hovertemplate='<b>Bola Pejal Utama</b><br>Jari-jari: R<sub>1</sub> = ' + str(R1) + '<br>Rapat Muatan: +ρ<extra></extra>'
    ))
    
    # 2. Arsiran rongga dalam bola (netral / tanpa muatan)
    fig.add_trace(go.Scatter(
        x=x_cav, y=y_cav,
        mode='lines',
        fill='toself',
        fillcolor='rgba(255, 255, 255, 0.98)',
        line=dict(color='#b91c1c', width=2.5, dash='dash'),
        name=f'Rongga Kosong, R<sub>2</sub>={R2}',
        hovertemplate='<b>Rongga Kosong</b><br>Pusat Rongga C: (' + str(a) + ', 0)<br>Jari-jari: R<sub>2</sub> = ' + str(R2) + '<extra></extra>'
    ))
    
    # 3. Sumbu koordinat x dan y
    margin = 1.25
    fig.add_trace(go.Scatter(
        x=[-R1*margin, R1*margin], y=[0, 0],
        mode='lines', line=dict(color='#94a3b8', width=1.2, dash='dot'),
        showlegend=False, hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=[0, 0], y=[-R1*margin, R1*margin],
        mode='lines', line=dict(color='#94a3b8', width=1.2, dash='dot'),
        showlegend=False, hoverinfo='skip'
    ))
    
    # 4. Titik pusat bola O dan pusat rongga C
    fig.add_trace(go.Scatter(
        x=[0, a], y=[0, 0],
        mode='markers+text',
        marker=dict(size=[13, 13], color=['#1e40af', '#dc2626']),
        text=['Pusat Bola O (0,0)', 'Pusat Rongga C (a,0)'],
        textposition=['bottom left', 'bottom right'],
        textfont=dict(size=12, color=['#1e40af', '#dc2626']),
        name='Titik Pusat Evaluasi',
        hovertemplate='<b>%{text}</b><br>Koordinat: (%{x:.2f}, %{y:.2f})<extra></extra>'
    ))
    
    # 5. Vektor medan listrik analitik di O dan C
    E0_mag = (rho * R2**3) / (3 * eps0 * a**2)
    Ecav_mag = (rho * a) / (3 * eps0)
    scale_E = 1.8
    
    # Trace dummy untuk entri legend vektor E
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines+markers',
        line=dict(color='#d97706', width=3),
        marker=dict(symbol='triangle-right', size=11),
        name=f'E<sub>pusat</sub> (arah +x) = {E0_mag:.3f} N/C'
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines+markers',
        line=dict(color='#ea580c', width=3),
        marker=dict(symbol='triangle-right', size=11),
        name=f'E<sub>rongga</sub> (arah +x) = {Ecav_mag:.3f} N/C'
    ))
    
    # Anotasi vektor pergeseran rongga a
    fig.add_annotation(
        x=a, y=0, ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.2, arrowwidth=3.2,
        arrowcolor='#059669',
        text='<b>a (offset)</b>',
        font=dict(color='#059669', size=13),
        yshift=18
    )
    
    # Anotasi jari-jari R1
    ang1 = np.pi * 0.38
    fig.add_annotation(
        x=R1*np.cos(ang1), y=R1*np.sin(ang1), ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.0, arrowwidth=1.8,
        arrowcolor='#1e3a8a',
        text='<b>R<sub>1</sub></b>',
        font=dict(color='#1e3a8a', size=13),
        xshift=-14, yshift=14
    )
    
    # Anotasi jari-jari R2
    ang2 = -np.pi * 0.4
    fig.add_annotation(
        x=a + R2*np.cos(ang2), y=R2*np.sin(ang2), ax=a, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.0, arrowwidth=1.8,
        arrowcolor='#991b1b',
        text='<b>R<sub>2</sub></b>',
        font=dict(color='#991b1b', size=13),
        xshift=14, yshift=-14
    )
    
    # Anotasi panah vektor medan E di O dan C
    fig.add_annotation(
        x=E0_mag * scale_E, y=0, ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=3, arrowsize=1.2, arrowwidth=3.2,
        arrowcolor='#d97706',
        text='<b>E<sub>pusat</sub></b>',
        font=dict(color='#d97706', size=11),
        yshift=24
    )
    
    fig.add_annotation(
        x=a + Ecav_mag * scale_E, y=0, ax=a, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=3, arrowsize=1.2, arrowwidth=3.2,
        arrowcolor='#ea580c',
        text='<b>E<sub>rongga</sub></b>',
        font=dict(color='#ea580c', size=11),
        yshift=24
    )
    
    fig.update_layout(
        title=dict(
            text='<b>Proyeksi Geometri Sistem Soal (Penampang Bidang z = 0) - Plotly 2D</b>',
            x=0.03, y=0.97, font=dict(size=15)
        ),
        xaxis=dict(
            title='Koordinat x',
            range=[-R1*margin, R1*margin],
            zeroline=False,
            gridcolor='#f1f5f9',
            scaleanchor='y',
            scaleratio=1
        ),
        yaxis=dict(
            title='Koordinat y',
            range=[-R1*margin, R1*margin],
            zeroline=False,
            gridcolor='#f1f5f9'
        ),
        template='plotly_white',
        width=850,
        height=780,
        legend=dict(
            x=0.02, y=0.02,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='#cbd5e1',
            borderwidth=1
        )
    )
    return fig

fig_2d = plot_proyeksi_sistem_2d()
fig_2d.show()

## 2. Perhitungan Simbolik Matematis (SymPy)

Kita gunakan Hukum Gauss pada masing-masing bola konstituen:

1. **Medan dan Potensial Bola Pejal Utuh Bermuatan Seragam $\rho$ Berjari-jari $R$:**
   - Di dalam bola ($r \le R$):
     $$\vec{E}_{dalam}(\vec{r}) = \frac{\rho}{3\varepsilon_0}\vec{r}$$
     $$V_{dalam}(r) = \frac{\rho}{6\varepsilon_0}\left(3R^2 - r^2\right)$$
   - Di luar bola ($r > R$):
     $$\vec{E}_{luar}(\vec{r}) = \frac{\rho R^3}{3\varepsilon_0 r^3}\vec{r} = \frac{Q}{4\pi\varepsilon_0 r^2}\hat{r}$$
     $$V_{luar}(r) = \frac{\rho R^3}{3\varepsilon_0 r} = \frac{Q}{4\pi\varepsilon_0 r}$$

2. **Pusat Bola Besar ($r = 0$):**
   - Jarak ke pusat rongga adalah $a > R_2$, sehingga titik origin berada di **luar** bola rongga negatif.
   - $\vec{E}_{pusat} = \vec{E}_1(0) + \vec{E}_2(-\vec{a}) = \vec{0} + \left[-\frac{(-\rho) R_2^3}{3\varepsilon_0 a^3}(-\vec{a})\right] = \frac{\rho R_2^3}{3\varepsilon_0 a^2} \hat{x}$
   - $V_{pusat} = V_{1, dalam}(0) + V_{2, luar}(a) = \frac{\rho R_1^2}{2\varepsilon_0} - \frac{\rho R_2^3}{3\varepsilon_0 a} = \frac{\rho}{2\varepsilon_0}\left(R_1^2 - \frac{2R_2^3}{3a}\right)$

3. **Pusat Rongga ($\vec{r} = \vec{a}$):**
   - Berada di dalam bola utama pada jarak $a$, dan tepat di pusat bola rongga (jarak 0 dari pusatnya).
   - $\vec{E}_{pusat\_rongga} = \frac{\rho}{3\varepsilon_0}\vec{a} + \vec{0} = \frac{\rho a}{3\varepsilon_0} \hat{x}$
   - $V_{pusat\_rongga} = \frac{\rho}{6\varepsilon_0}\left(3R_1^2 - a^2\right) - \frac{\rho R_2^2}{2\varepsilon_0} = \frac{\rho}{6\varepsilon_0}\left(3R_1^2 - 3R_2^2 - a^2\right)$

4. **Teorema Menarik: Medan di Seluruh Titik Dalam Rongga Bersifat Homogen!**
   $$\vec{E}_{rongga}(\vec{r}) = \frac{\rho}{3\varepsilon_0}\vec{r} + \left[\frac{-\rho}{3\varepsilon_0}(\vec{r} - \vec{a})\right] = \frac{\rho}{3\varepsilon_0}\vec{a} = \text{konstan!}$$

## 3. Proyeksi Sistem & Hasil Medan/Potensial Menggunakan Plot 3D Interaktif (Plotly)

Untuk memenuhi kebutuhan visualisasi spasial 3D interaktif, modul berikut memproyeksikan:
1. **Model Geometri Sistem 3D (Cutaway Projection)**: Memperlihatkan bola utama semi-transparan dengan bukaan oktan untuk menyingkap posisi rongga bola kecil di dalamnya beserta sumbu koordinat dan vektor jarak $\vec{a}$.
2. **Vektor Medan Listrik 3D (Electric Field Cones)**: Vektor panah 3D $\vec{E}(x,y,z)$ di dalam dan luar bola, dengan sorotan khusus medan homogen di dalam rongga.
3. **Titik Evaluasi Khusus**: Menampilkan letak titik $r=0$ dan $r=a$ beserta vektor medan listriknya.
4. **Permukaan Ekipotensial 3D (Isosurface)**: Menampilkan lapisan-lapisan permukaan potensial konstan $V(x,y,z)$ di sekitar bola dan rongga.
5. **Menu Kontrol Interaktif**: Terdapat dropdown di pojok grafik untuk beralih antara tampilan geometri sistem, medan listrik, potensial, dan semua elemen secara dinamis.

In [15]:
import plotly.graph_objects as go

# Fungsi umum medan dan potensial bola konstituen
def medan_dan_potensial_bola(points, center, radius, rapat_muatan, eps0=1.0):
    disp = points - center
    r = np.linalg.norm(disp, axis=-1)
    r_safe = np.maximum(r, 1e-12)
    
    inside = r <= radius
    outside = ~inside
    
    # Medan Listrik
    E = np.zeros_like(disp)
    E[inside] = (rapat_muatan / (3.0 * eps0)) * disp[inside]
    E[outside] = (rapat_muatan * radius**3 / (3.0 * eps0)) * (disp[outside] / (r_safe[outside, None]**3))
    
    # Potensial Listrik
    V = np.zeros(r.shape, dtype=float)
    V[inside] = (rapat_muatan / (6.0 * eps0)) * (3.0 * radius**2 - r[inside]**2)
    V[outside] = (rapat_muatan * radius**3 / (3.0 * eps0)) / r_safe[outside]
    return E, V

def visualisasi_3d_interaktif(R1=3.0, R2=1.0, a=1.4, rho=1.0, eps0=1.0):
    # --- 1. Permukaan Geometri Sistem 3D ---
    u = np.linspace(0, 2*np.pi, 50)
    v = np.linspace(0, np.pi, 25)
    su = np.outer(np.cos(u), np.sin(v))
    sv = np.outer(np.sin(u), np.sin(v))
    sw = np.outer(np.ones_like(u), np.cos(v))
    
    # Bola utama dengan potongan cutaway oktan (x>0 dan y>0) untuk melihat rongga di dalam
    X_out, Y_out, Z_out = R1 * su, R1 * sv, R1 * sw
    cutaway = (X_out > 0) & (Y_out > 0)
    X_cut = np.where(cutaway, np.nan, X_out)
    Y_cut = np.where(cutaway, np.nan, Y_out)
    Z_cut = np.where(cutaway, np.nan, Z_out)
    
    surf_outer = go.Surface(
        x=X_cut, y=Y_cut, z=Z_cut,
        opacity=0.28, colorscale=[[0, '#3b82f6'], [1, '#1d4ed8']],
        showscale=False, name=f'Bola Utama (R1={R1})',
        hovertemplate='Bola Utama Pejal: R1 = %{x:.2f}<extra></extra>'
    )
    
    # Rongga dalam bola
    X_cav = a + R2 * su
    Y_cav = R2 * sv
    Z_cav = R2 * sw
    surf_cav = go.Surface(
        x=X_cav, y=Y_cav, z=Z_cav,
        opacity=0.82, colorscale=[[0, '#ef4444'], [1, '#991b1b']],
        showscale=False, name=f'Dinding Rongga (R2={R2})',
        hovertemplate='Dinding Rongga: r = %{x:.2f}<extra></extra>'
    )
    
    # Penanda titik pusat bola dan pusat rongga
    centers = go.Scatter3d(
        x=[0, a], y=[0, 0], z=[0, 0],
        mode='markers+text',
        marker=dict(size=[9, 9], color=['#1e40af', '#dc2626']),
        text=['Pusat Bola O (r=0)', 'Pusat Rongga C (r=a)'],
        textposition='top center',
        name='Titik Pusat Evaluasi'
    )
    
    # Garis vektor pergeseran a
    line_a = go.Scatter3d(
        x=[0, a], y=[0, 0], z=[0, 0],
        mode='lines',
        line=dict(color='#059669', width=6, dash='solid'),
        name='Vektor Pergeseran a'
    )
    
    # --- 2. Vektor Medan Listrik 3D (Cones) ---
    N_grid = 9
    coords = np.linspace(-R1*1.15, R1*1.15, N_grid)
    Xg, Yg, Zg = np.meshgrid(coords, coords, coords, indexing='ij')
    pts_field = np.column_stack((Xg.ravel(), Yg.ravel(), Zg.ravel()))
    
    E1, _ = medan_dan_potensial_bola(pts_field, np.array([0,0,0]), R1, rho, eps0)
    E2, _ = medan_dan_potensial_bola(pts_field, np.array([a,0,0]), R2, -rho, eps0)
    E_tot = E1 + E2
    E_mag = np.linalg.norm(E_tot, axis=1)
    
    mask_field = (E_mag > 0.04) & (E_mag < np.percentile(E_mag, 97))
    field_cones = go.Cone(
        x=pts_field[mask_field, 0], y=pts_field[mask_field, 1], z=pts_field[mask_field, 2],
        u=E_tot[mask_field, 0], v=E_tot[mask_field, 1], w=E_tot[mask_field, 2],
        colorscale='Turbo', cmin=0, cmax=float(np.percentile(E_mag, 95)),
        colorbar=dict(title="|E| (N/C)", len=0.65, x=1.02),
        sizemode='scaled', sizeref=1.6, anchor='tail',
        name='Medan Listrik E (3D)', showscale=True
    )
    
    # Vektor E khusus di titik evaluasi: O(0,0,0) dan C(a,0,0)
    pts_eval = np.array([[0.0, 0.0, 0.0], [a, 0.0, 0.0]])
    E1_ev, _ = medan_dan_potensial_bola(pts_eval, np.array([0,0,0]), R1, rho, eps0)
    E2_ev, _ = medan_dan_potensial_bola(pts_eval, np.array([a,0,0]), R2, -rho, eps0)
    E_eval_tot = E1_ev + E2_ev
    
    eval_cones = go.Cone(
        x=[0, a], y=[0, 0], z=[0, 0],
        u=[E_eval_tot[0, 0], E_eval_tot[1, 0]],
        v=[E_eval_tot[0, 1], E_eval_tot[1, 1]],
        w=[E_eval_tot[0, 2], E_eval_tot[1, 2]],
        colorscale=[[0, '#f59e0b'], [1, '#d97706']],
        sizemode='absolute', sizeref=0.55, anchor='tail',
        showscale=False, name='E di Titik Soal (O & C)'
    )
    
    # --- 3. Isosurface Potensial Listrik 3D ---
    N_iso = 21
    coords_iso = np.linspace(-R1*1.1, R1*1.1, N_iso)
    Xi, Yi, Zi = np.meshgrid(coords_iso, coords_iso, coords_iso, indexing='ij')
    pts_iso = np.column_stack((Xi.ravel(), Yi.ravel(), Zi.ravel()))
    
    _, V1_iso = medan_dan_potensial_bola(pts_iso, np.array([0,0,0]), R1, rho, eps0)
    _, V2_iso = medan_dan_potensial_bola(pts_iso, np.array([a,0,0]), R2, -rho, eps0)
    V_tot = (V1_iso + V2_iso).reshape(Xi.shape)
    
    iso_potential = go.Isosurface(
        x=Xi.ravel(), y=Yi.ravel(), z=Zi.ravel(), value=V_tot.ravel(),
        isomin=float(np.percentile(V_tot, 25)),
        isomax=float(np.percentile(V_tot, 92)),
        surface_count=5, opacity=0.45, colorscale='Viridis',
        colorbar=dict(title="V (Volt)", len=0.65, x=1.12),
        slices_z=dict(show=True, locations=[0]),
        name='Potensial Listrik 3D (Isosurface)', showscale=True,
        visible=False
    )
    
    # Urutan traces:
    # 0: surf_outer, 1: surf_cav, 2: centers, 3: line_a
    # 4: field_cones, 5: eval_cones
    # 6: iso_potential
    fig = go.Figure(data=[surf_outer, surf_cav, centers, line_a, field_cones, eval_cones, iso_potential])
    
    # Menu Dropdown Interaktif untuk kontrol tampilan
    fig.update_layout(
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                x=0.01, y=1.15,
                showactive=True,
                buttons=[
                    dict(
                        label="Gabungan (Geometri Sistem + Medan E)",
                        method="update",
                        args=[{"visible": [True, True, True, True, True, True, False]}]
                    ),
                    dict(
                        label="Proyeksi Geometri Sistem Soal 3D",
                        method="update",
                        args=[{"visible": [True, True, True, True, False, False, False]}]
                    ),
                    dict(
                        label="Vektor Medan Listrik 3D (E Cones)",
                        method="update",
                        args=[{"visible": [False, True, True, True, True, True, False]}]
                    ),
                    dict(
                        label="Potensial Listrik 3D (Isosurface)",
                        method="update",
                        args=[{"visible": [False, True, True, True, False, False, True]}]
                    ),
                    dict(
                        label="Tampilkan Semua Komponen Lengkap",
                        method="update",
                        args=[{"visible": [True, True, True, True, True, True, True]}]
                    )
                ]
            )
        ],
        title=dict(
            text="<b>Proyeksi 3D Interaktif: Geometri Sistem & Medan/Potensial Bola Berongga</b>",
            x=0.02, y=0.98
        ),
        template="plotly_white",
        width=950, height=720,
        scene=dict(
            xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
            aspectmode="cube",
            camera=dict(eye=dict(x=1.8, y=1.8, z=1.25))
        ),
        margin=dict(l=0, r=0, t=60, b=0),
        legend=dict(x=0.02, y=0.92, bgcolor='rgba(255,255,255,0.7)')
    )
    return fig

fig_3d = visualisasi_3d_interaktif(R1_def, R2_def, a_def, rho_def, eps0_def)
fig_3d.show()

## 4. Proyeksi Topografi Permukaan Potensial 3D (Landscape $V(x,y)$)

Proyeksi permukaan elevasi 3D dari potensial skalar $V(x,y)$ pada bidang ekuator ($z=0$) memberikan analogi intuitif seperti "bukit potensial" bola pejal dan "cekungan/lembah" yang dibentuk oleh rongga bermuatan negatif efektif.

In [16]:
def plot_landscape_potensial_3d(R1=3.0, R2=1.0, a=1.4, rho=1.0, eps0=1.0):
    """Membuat plot 3D surface relief untuk potensial listrik pada bidang z = 0."""
    N_surf = 120
    x_range = np.linspace(-R1*1.3, R1*1.3, N_surf)
    y_range = np.linspace(-R1*1.3, R1*1.3, N_surf)
    Xs, Ys = np.meshgrid(x_range, y_range)
    pts_surf = np.column_stack((Xs.ravel(), Ys.ravel(), np.zeros(Xs.size)))
    
    _, V1_s = medan_dan_potensial_bola(pts_surf, np.array([0,0,0]), R1, rho, eps0)
    _, V2_s = medan_dan_potensial_bola(pts_surf, np.array([a,0,0]), R2, -rho, eps0)
    Zs = (V1_s + V2_s).reshape(Xs.shape)
    
    fig_surf = go.Figure(data=[
        go.Surface(
            x=Xs, y=Ys, z=Zs,
            colorscale='Viridis',
            colorbar=dict(title='V (Volt)', len=0.7),
            contours_z=dict(show=True, usecolormap=True, highlightcolor="limegreen", project_z=True),
            hovertemplate='x: %{x:.2f}<br>y: %{y:.2f}<br>V: %{z:.3f} V<extra></extra>'
        )
    ])
    
    fig_surf.update_layout(
        title='<b>Proyeksi 3D Permukaan Kontur Potensial V(x, y) pada Bidang z = 0</b>',
        template='plotly_white',
        width=950, height=680,
        scene=dict(
            xaxis_title='x',
            yaxis_title='y',
            zaxis_title='Potensial V (Volt)',
            camera=dict(eye=dict(x=-1.6, y=-1.6, z=1.3))
        ),
        margin=dict(l=0, r=0, t=50, b=0)
    )
    return fig_surf

fig_landscape = plot_landscape_potensial_3d(R1_def, R2_def, a_def, rho_def, eps0_def)
fig_landscape.show()

## 5. Profil 1D Analitik Sepanjang Sumbu Rongga (Sumbu $x$)

Untuk memvalidasi secara kuantitatif hasil perhitungan analitik di pusat bola ($x=0$) dan di pusat rongga ($x=a$), kita plot kurva medan listrik komponen $E_x(x)$ dan potensial listrik $V(x)$ di sepanjang sumbu $x$ ($y=0, z=0$) menggunakan **Plotly 2D Subplots**.

> **Fakta Fisika Utama:** Perhatikan bahwa di dalam rentang rongga ($a - R_2 \le x \le a + R_2$), kurva medan $E_x(x)$ bernilai **horizontal sempurna (konstan)** sebesar $\frac{\rho a}{3\varepsilon_0}$!

In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_profil_1d_sumbu_x(R1=3.0, R2=1.0, a=1.4, rho=1.0, eps0=1.0):
    """Membuat grafik profil 1D analitik sepanjang sumbu x menggunakan Plotly subplots."""
    x_vals = np.linspace(-R1*1.4, R1*1.4, 600)
    pts_1d = np.column_stack((x_vals, np.zeros_like(x_vals), np.zeros_like(x_vals)))
    
    E1_1d, V1_1d = medan_dan_potensial_bola(pts_1d, np.array([0,0,0]), R1, rho, eps0)
    E2_1d, V2_1d = medan_dan_potensial_bola(pts_1d, np.array([a,0,0]), R2, -rho, eps0)
    
    E_x_tot = E1_1d[:, 0] + E2_1d[:, 0]
    V_tot = V1_1d + V2_1d
    
    E0_val = (rho * R2**3) / (3 * eps0 * a**2)
    Ea_val = (rho * a) / (3 * eps0)
    V0_val = (rho / (2 * eps0)) * (R1**2 - (2 * R2**3) / (3 * a))
    Va_val = (rho / (6 * eps0)) * (3 * R1**2 - 3 * R2**2 - a**2)
    
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=(
            '<b>Profil Medan Listrik E<sub>x</sub>(x) Melintasi Sumbu Rongga</b>',
            '<b>Profil Potensial Listrik V(x) Melintasi Sumbu Rongga</b>'
        )
    )
    
    # Daerah bola pejal dan rongga diarsir
    for r in [1, 2]:
        fig.add_vrect(
            x0=-R1, x1=R1,
            fillcolor='#eff6ff', opacity=0.7,
            layer='below', line_width=0,
            row=r, col=1
        )
        fig.add_vrect(
            x0=a - R2, x1=a + R2,
            fillcolor='#fee2e2', opacity=0.85,
            layer='below', line_width=1.2, line_dash='dash', line_color='#f87171',
            row=r, col=1
        )
        fig.add_vline(x=0, line_dash='dash', line_color='#1e40af', line_width=1.2, row=r, col=1)
        fig.add_vline(x=a, line_dash='dash', line_color='#dc2626', line_width=1.2, row=r, col=1)
    
    # 1. Kurva Ex(x)
    fig.add_trace(go.Scatter(
        x=x_vals, y=E_x_tot,
        mode='lines',
        line=dict(color='#d97706', width=2.6),
        name='E<sub>x</sub>(x) Total',
        hovertemplate='x: %{x:.3f}<br>E<sub>x</sub>: %{y:.4f} N/C<extra></extra>'
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=[0, a], y=[E0_val, Ea_val],
        mode='markers',
        marker=dict(size=10, color=['#1e40af', '#dc2626']),
        name='Titik Evaluasi Medan (O & C)',
        hovertemplate='Titik: (%{x:.2f}, %{y:.4f})<extra></extra>'
    ), row=1, col=1)
    
    # 2. Kurva V(x)
    fig.add_trace(go.Scatter(
        x=x_vals, y=V_tot,
        mode='lines',
        line=dict(color='#059669', width=2.6),
        name='V(x) Total',
        hovertemplate='x: %{x:.3f}<br>V: %{y:.4f} Volt<extra></extra>'
    ), row=2, col=1)
    
    fig.add_trace(go.Scatter(
        x=[0, a], y=[V0_val, Va_val],
        mode='markers',
        marker=dict(size=10, color=['#1e40af', '#dc2626']),
        name='Titik Evaluasi Potensial (O & C)',
        hovertemplate='Titik: (%{x:.2f}, %{y:.4f})<extra></extra>'
    ), row=2, col=1)
    
    # Anotasi nilai analitik
    fig.add_annotation(
        x=0, y=E0_val, xref='x1', yref='y1',
        text=f'<b>E(0) = {E0_val:.3f}</b>',
        showarrow=True, arrowhead=2, arrowcolor='#1e40af',
        ax=-50, ay=-30, font=dict(color='#1e40af', size=11)
    )
    fig.add_annotation(
        x=a, y=Ea_val, xref='x1', yref='y1',
        text=f'<b>E(a) = {Ea_val:.3f} (konstan)</b>',
        showarrow=True, arrowhead=2, arrowcolor='#dc2626',
        ax=65, ay=-30, font=dict(color='#dc2626', size=11)
    )
    fig.add_annotation(
        x=0, y=V0_val, xref='x2', yref='y2',
        text=f'<b>V(0) = {V0_val:.3f} V</b>',
        showarrow=True, arrowhead=2, arrowcolor='#1e40af',
        ax=-55, ay=35, font=dict(color='#1e40af', size=11)
    )
    fig.add_annotation(
        x=a, y=Va_val, xref='x2', yref='y2',
        text=f'<b>V(a) = {Va_val:.3f} V</b>',
        showarrow=True, arrowhead=2, arrowcolor='#dc2626',
        ax=55, ay=35, font=dict(color='#dc2626', size=11)
    )
    
    fig.update_yaxes(title_text='<b>Medan Listrik E<sub>x</sub> (N/C)</b>', row=1, col=1, gridcolor='#f1f5f9')
    fig.update_yaxes(title_text='<b>Potensial Listrik V (Volt)</b>', row=2, col=1, gridcolor='#f1f5f9')
    fig.update_xaxes(title_text='<b>Koordinat x</b>', row=2, col=1, gridcolor='#f1f5f9')
    fig.update_xaxes(gridcolor='#f1f5f9', row=1, col=1)
    
    fig.update_layout(
        template='plotly_white',
        width=950,
        height=750,
        hovermode='closest',
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255, 255, 255, 0.9)', bordercolor='#cbd5e1', borderwidth=1),
        margin=dict(l=60, r=40, t=60, b=50)
    )
    return fig

fig_1d = plot_profil_1d_sumbu_x(R1_def, R2_def, a_def, rho_def, eps0_def)
fig_1d.show()

## 6. Eksplorasi Parameter Dinamis (Interactive Slider Widget)

Gunakan slider interaktif di bawah ini untuk mengubah jari-jari bola $R_1$, jari-jari rongga $R_2$, jarak offset $a$, dan rapat muatan $
ho$ untuk mengeksplorasi perubahan nilai medan serta bentuk proyeksi sistem secara dinamis dengan Plotly.

In [18]:
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets

def eksplorasi_interaktif(R1=3.0, R2=1.0, a=1.4, rho=1.0):
    """Fungsi widget interaktif untuk mengeksplorasi parameter fisika secara dinamis."""
    if a + R2 >= R1:
        print("⚠️ Peringatan Fisika: Rongga harus berada sepenuhnya di dalam bola utama (a + R2 < R1)!")
        return
    
    E0 = (rho * R2**3) / (3.0 * 1.0 * a**2)
    Ea = (rho * a) / (3.0 * 1.0)
    V0 = (rho / 2.0) * (R1**2 - (2.0 * R2**3) / (3.0 * a))
    Va = (rho / 6.0) * (3.0 * R1**2 - 3.0 * R2**2 - a**2)
    
    print(f"--- Evaluasi Nilai Fisik Sistem (R1={R1:.2f}, R2={R2:.2f}, a={a:.2f}, rho={rho:.2f}) ---")
    print(f"• Medan di Pusat Bola E(0)       = {E0:.4f} N/C (arah +x)")
    print(f"• Medan di Pusat Rongga E(a)     = {Ea:.4f} N/C (arah +x, homogen di dalam rongga)")
    print(f"• Potensial di Pusat Bola V(0)   = {V0:.4f} Volt")
    print(f"• Potensial di Pusat Rongga V(a) = {Va:.4f} Volt")
    print("-" * 75)
    
    fig = plot_proyeksi_sistem_2d(R1, R2, a, rho)
    fig.show()

interact(
    eksplorasi_interaktif,
    R1=FloatSlider(value=3.0, min=2.0, max=5.0, step=0.1, description='R1 (Bola):'),
    R2=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='R2 (Rongga):'),
    a=FloatSlider(value=1.4, min=0.3, max=3.0, step=0.1, description='a (Offset):'),
    rho=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='rho (Muatan):')
);

interactive(children=(FloatSlider(value=3.0, description='R1 (Bola):', max=5.0, min=2.0), FloatSlider(value=1.…

from ipywidgets import interact, FloatSlider
import ipywidgets as widgets

def eksplorasi_interaktif(R1=3.0, R2=1.0, a=1.4, rho=1.0):
    """Fungsi widget interaktif untuk mengeksplorasi parameter fisika secara dinamis."""
    if a + R2 >= R1:
        print("⚠️ Peringatan Fisika: Rongga harus berada sepenuhnya di dalam bola utama (a + R2 < R1)!")
        return
    
    E0 = (rho * R2**3) / (3.0 * 1.0 * a**2)
    Ea = (rho * a) / (3.0 * 1.0)
    V0 = (rho / 2.0) * (R1**2 - (2.0 * R2**3) / (3.0 * a))
    Va = (rho / 6.0) * (3.0 * R1**2 - 3.0 * R2**2 - a**2)
    
    print(f"--- Evaluasi Nilai Fisik Sistem (R1={R1:.2f}, R2={R2:.2f}, a={a:.2f}, rho={rho:.2f}) ---")
    print(f"• Medan di Pusat Bola E(0)       = {E0:.4f} N/C (arah +x)")
    print(f"• Medan di Pusat Rongga E(a)     = {Ea:.4f} N/C (arah +x, homogen di dalam rongga)")
    print(f"• Potensial di Pusat Bola V(0)   = {V0:.4f} Volt")
    print(f"• Potensial di Pusat Rongga V(a) = {Va:.4f} Volt")
    print("-" * 75)
    
    fig = plot_proyeksi_sistem_2d(R1, R2, a, rho)
    fig.show()

interact(
    eksplorasi_interaktif,
    R1=FloatSlider(value=3.0, min=2.0, max=5.0, step=0.1, description='R1 (Bola):'),
    R2=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='R2 (Rongga):'),
    a=FloatSlider(value=1.4, min=0.3, max=3.0, step=0.1, description='a (Offset):'),
    rho=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='rho (Muatan):')
);